# 2. Prepare the modelling data

This notebook standardises a small set of tender fields and creates features that describe the tender before an award. It also records which fields are not used because they are only known after the tender has progressed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if not (PROJECT / "data").exists():
    PROJECT = PROJECT.parent
INPUT_FILE = PROJECT / "data" / "processed" / "ppip_processes.csv"
OUTPUT_FILE = PROJECT / "data" / "processed" / "ppip_features.csv"
REPORTS = PROJECT / "reports"
REPORTS.mkdir(exist_ok=True)

raw = pd.read_csv(INPUT_FILE, parse_dates=["tender_start_date", "tender_end_date", "record_date"])
print("Processes loaded:", f"{len(raw):,}")
display(raw.head())

## Clean the fields used in the model

Missing amounts and durations remain missing. Amounts in currencies other than KES are not converted without an exchange-rate source. Entity names and identifiers are kept for reference but are not model inputs.

In [ ]:
METHOD_MAP = {
    "open": "open",
    "selective": "selective",
    "restricted": "selective",
    "direct": "direct",
    "limited": "limited",
    "quotation": "direct",
    "requestforquotation": "direct",
}
CATEGORY_MAP = {
    "goods": "goods",
    "services": "services",
    "works": "works",
    "consulting": "consulting",
    "consultancy": "consulting",
}
COUNTY_ALIASES = {
    "nrb": "nairobi", "nbo": "nairobi", "nairobi city": "nairobi",
    "msa": "mombasa", "ksm": "kisumu", "nkr": "nakuru",
}

def normalise(value, mapping):
    key = str(value or "unknown").strip().lower().replace(" ", "").replace("-", "")
    return mapping.get(key, "unknown")


def normalise_county(value):
    key = str(value or "unknown").strip().lower()
    return COUNTY_ALIASES.get(key, key.replace(" ", "_") if key else "unknown")


data = raw.copy()
data["tender_start_date"] = pd.to_datetime(data["tender_start_date"], utc=True, errors="coerce")
data["estimated_value_kes"] = pd.to_numeric(data["estimated_value_kes"], errors="coerce")
data["duration_days"] = pd.to_numeric(data["duration_days"], errors="coerce")
data["num_tenderers"] = pd.to_numeric(data["num_tenderers"], errors="coerce")
data["procurement_method"] = data["procurement_method"].map(lambda x: normalise(x, METHOD_MAP))
data["procurement_category"] = data["procurement_category"].map(lambda x: normalise(x, CATEGORY_MAP))
data["county"] = data["county_raw"].map(normalise_county)
data["tender_text"] = data["tender_text"].fillna("").astype(str).str.strip()
data["value_missing"] = data["estimated_value_kes"].isna().astype(int)
data["duration_missing"] = data["duration_days"].isna().astype(int)
data["tenderers_missing"] = data["num_tenderers"].isna().astype(int)
data["text_missing"] = data["tender_text"].eq("").astype(int)
data["estimated_value_log1p"] = np.log1p(data["estimated_value_kes"].where(data["estimated_value_kes"] > 0))
month = data["tender_start_date"].dt.month
day = data["tender_start_date"].dt.dayofweek
data["month_sin"] = np.sin(2 * np.pi * month / 12)
data["month_cos"] = np.cos(2 * np.pi * month / 12)
data["weekday_sin"] = np.sin(2 * np.pi * day / 7)
data["weekday_cos"] = np.cos(2 * np.pi * day / 7)

## Set the feature list

The model does not use award suppliers, contract values, contract dates, award results or the current tender status. The public download is a compiled snapshot, so the tender start date is used only to order the records for a chronological check; it is not a verified publication timestamp.

In [ ]:
NUMERIC_CANDIDATES = [
    "estimated_value_log1p", "duration_days", "num_tenderers",
    "month_sin", "month_cos", "weekday_sin", "weekday_cos",
    "value_missing", "duration_missing", "tenderers_missing", "text_missing",
]
CATEGORICAL_CANDIDATES = ["procurement_method", "procurement_category", "county"]

NUMERIC_COLUMNS = [
    col for col in NUMERIC_CANDIDATES
    if col in data.columns and data[col].notna().any() and data[col].nunique(dropna=True) > 1
]
CATEGORICAL_COLUMNS = [
    col for col in CATEGORICAL_CANDIDATES
    if col in data.columns and data[col].nunique(dropna=True) > 1
]
INPUT_COLUMNS = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
if data["tender_text"].str.strip().ne("").sum() >= 20:
    INPUT_COLUMNS.append("tender_text")

prepared = data.dropna(subset=["tender_start_date"]).copy()
prepared = prepared.sort_values(["tender_start_date", "ocid"]).reset_index(drop=True)
prepared = prepared[["ocid", "tender_start_date"] + INPUT_COLUMNS]
if len(prepared) < 100:
    raise ValueError(f"Only {len(prepared)} dated processes are available. Check the source data before modelling.")
if not INPUT_COLUMNS:
    raise ValueError("No usable tender-time features were found.")
prepared.to_csv(OUTPUT_FILE, index=False)

feature_audit = pd.DataFrame({
    "feature": INPUT_COLUMNS,
    "type": ["numeric" if col in NUMERIC_COLUMNS else "categorical/text" for col in INPUT_COLUMNS],
    "available_pct": [round(prepared[col].notna().mean() * 100, 2) if col != "tender_text" else round(prepared[col].fillna("").ne("").mean() * 100, 2) for col in INPUT_COLUMNS],
})
feature_audit.to_csv(REPORTS / "feature_audit.csv", index=False)
print("Rows kept for modelling:", f"{len(prepared):,}")
print("Numeric features:", NUMERIC_COLUMNS)
print("Categorical features:", CATEGORICAL_COLUMNS)
print("Text included:", "tender_text" in INPUT_COLUMNS)
display(feature_audit)
display(prepared.head())